# Bigram v0 on handwriting labels

A **bigram** is a token for two adjacent letters. Both vocabularies below add 100 bigrams to the same 59 character fallbacks and CTC blank. The **force-ranked** version picks pairs with the highest provisional pen-contact scores. The **frequency-ranked** version picks the most common pairs in training recordings. Both choose from the same 176 eligible pairs in OnHW writer-independent, right-handed fold 0.

Run this notebook from the TVA repository root with the TVA Python environment. It loads the saved fold-0 vocabularies. See the [forced-alignment review](docs/FORCED_ALIGNMENT.md) for the build command and selection details.

In [1]:
from pathlib import Path
import json

from IPython.display import Markdown, display
from tva.tokenizers import get_tokenizer

run = Path("results/thesis/forced_alignment_review/wi_rh_fold0_20261117_bigram_v0")
tokenizers = {}
for label, variant in (("Force-ranked", "force"), ("Frequency-ranked", "frequency")):
    vocab_path = run / variant / "0.json"
    if not vocab_path.is_file():
        raise FileNotFoundError(f"Missing {vocab_path}. See the Bigram v0 build command in docs/FORCED_ALIGNMENT.md.")
    tokenizer = get_tokenizer("bigram")
    tokenizer.load(vocab_path)
    tokenizers[label] = tokenizer

2026-11-17 22:49:47.469 | INFO     | tva.tokenizers:load:214 - BigramTokenizer is loaded from results/thesis/forced_alignment_review/wi_rh_fold0_20261117_bigram_v0/force/0.json.


2026-11-17 22:49:47.472 | INFO     | tva.tokenizers:load:214 - BigramTokenizer is loaded from results/thesis/forced_alignment_review/wi_rh_fold0_20261117_bigram_v0/frequency/0.json.


## How a word is split

The tokenizer scans left to right. If the next two letters are in its vocabulary, it emits their bigram and advances two letters. Otherwise it emits one letter and advances one.

For `Mittwoch`, the force vocabulary starts with `Mi` and next checks `tt`. The frequency vocabulary starts with `M` and next checks `it`. The first choice changes which pair the tokenizer can consider next.

In [2]:
def split_into_tokens(tokenizer, word):
    return [tokenizer.idx_token[token_id] for token_id in tokenizer.encode(word)]


words = ["Mittwoch", "drei", "Programm", "Chief", "Haus", "schön"]
rows = [
    "| Word | Characters | Force-ranked | Frequency-ranked |",
    "| --- | --- | --- | --- |",
]
for word in words:
    chunks = [" · ".join(word)]
    chunks += [" · ".join(split_into_tokens(t, word)) for t in tokenizers.values()]
    rows.append("| " + " | ".join([word, *chunks]) + " |")

display(Markdown("\n".join(rows)))

| Word | Characters | Force-ranked | Frequency-ranked |
| --- | --- | --- | --- |
| Mittwoch | M · i · t · t · w · o · c · h | Mi · tt · wo · ch | M · it · t · wo · ch |
| drei | d · r · e · i | dr · ei | d · re · i |
| Programm | P · r · o · g · r · a · m · m | Pr · o · gr · am · m | P · ro · g · ra · mm |
| Chief | C · h · i · e · f | C · hi · e · f | C · h · ie · f |
| Haus | H · a · u · s | H · au · s | H · au · s |
| schön | s · c · h · ö · n | s · ch · ön | sc · h · ö · n |

## Try a word

Edit `word` below and rerun the cell to see its tokens, their IDs and the decoded text. Use letters in the OnHW alphabet (A–Z, a–z, ÄÖÜ, äöü and ß). Case matters. Bigram IDs are assigned separately in each vocabulary, so use the same tokenizer to encode and decode.

In [3]:
word = "Mittwoch"

for label, tokenizer in tokenizers.items():
    ids = tokenizer.encode(word)
    decoded = tokenizer.decode(ids)
    assert decoded == word
    print(label)
    print("  Tokens", [tokenizer.idx_token[token_id] for token_id in ids])
    print("  IDs   ", ids)
    print("  Text  ", decoded)

Force-ranked
  Tokens ['Mi', 'tt', 'wo', 'ch']
  IDs    [60, 151, 87, 132]
  Text   Mittwoch
Frequency-ranked
  Tokens ['M', 'it', 't', 'wo', 'ch']
  IDs    [13, 76, 49, 156, 63]
  Text   Mittwoch


## Across the training fold

The saved training summary counts tokens over all 19,907 recordings. Shorter targets mean fewer tokens to emit. Recognition accuracy requires a separate evaluation.

In [4]:
summary = json.loads((run / "summary.json").read_text(encoding="utf-8"))

print(f"Shared bigrams          {summary['shared_pairs']} of 100")
print(f"Character target count  {summary['characters']:,}")
for label, stats in summary["conditions"].items():
    print(
        f"{label.capitalize():<24}{stats['tokens']:,} tokens"
        f" ({stats['token_reduction_fraction']:.1%} fewer)"
    )

Shared bigrams          58 of 100
Character target count  108,199
Force                   75,766 tokens (30.0% fewer)
Frequency               70,841 tokens (34.5% fewer)
